# Financial Fraud Detection Using Network Graph Analytics and ML

**Domain:** Finance / Fraud Analytics / Data Science  
**Primary Evaluation Focus:** F1 Score, AUC-ROC, Recall  
**Dataset:** PaySim-style transaction data

##  Executive Summary

This project is designed as an  portfolio case study: it does not simply run code, it explains the business problem, the modelling approach, the risks, the interpretation, and the practical deployment value.

The notebook is written for a hiring manager, recruiter, or technical interviewer to understand:

- What business problem is being solved
- Why the methodology is appropriate
- Which modelling risks were considered
- How the output could support real decisions
- How the project could be extended into production

## Key Findings

- Fraud detection should prioritise recall and F1 Score rather than accuracy because fraud is rare.
- Graph-based customer and account connectivity can expose suspicious transaction structures.
- Transaction amount, account movement and network behaviour are stronger indicators than raw transaction type alone.
- NetworkX features make the project stand out beyond a standard tabular classifier.
- False negatives carry high financial and regulatory cost in fraud workflows.

## Business Recommendations

- Use graph-based features to flag mule-account and money-ring behaviour.
- Set operating thresholds based on investigation capacity and fraud loss appetite.
- Deploy a tiered alerting workflow for high-risk transactions.
- Monitor feature drift as fraud patterns change over time.
- Add analyst feedback loops to improve model retraining.

## 

This  places emphasis on:

- Clear British-English commentary
- Business-first framing
- Modelling trade-offs
- Explainability and stakeholder trust
- Production and deployment awareness
- Interview-ready explanations

# Methodology and Modelling Rationale

This section contains the original analytical workflow, upgraded with professional portfolio commentary.

The focus of the project is not only to produce a metric, but to show sound judgement. In a commercial data role, the strongest candidates are able to explain:

- Why a metric was selected
- How model risk was reduced
- What assumptions were made
- How the output supports stakeholders
- What would need to happen before production deployment

For this project, the primary evaluation focus is: **F1 Score, AUC-ROC, Recall**.

# Project 02 - Financial Fraud Detection: Graph Networks
**Domain:** Finance / Data Science | **Key Techniques:** NetworkX Graph Analysis + 12-Classifier Ensemble

Detects fraudulent transactions across 50,000 accounts using network graphs to uncover money rings.

**Dataset:** [PaySim on Kaggle](https://www.kaggle.com/datasets/ealaxi/paysim1) — save as `fraud_transactions.csv`

In [ ]:
# Install: pip install scikit-learn xgboost lightgbm networkx pandas matplotlib seaborn scipy
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, roc_auc_score, confusion_matrix, roc_curve, classification_report
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
print("All libraries loaded")

## 1. Load Data

In [ ]:
df = pd.read_csv('fraud_transactions.csv')
print(f"Shape: {df.shape}")
print(f"Fraud rate: {df['isFraud'].mean():.4f} ({df['isFraud'].sum():,} fraudulent)")
print(f"Transaction types: {df['type'].value_counts().to_dict()}")
df.head()

## 2. EDA

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

df.groupby('type')['isFraud'].mean().sort_values(ascending=False).plot(
    kind='bar', ax=axes[0], color='#C9A96E', edgecolor='#0F1C2E')
axes[0].set_title('Fraud Rate by Transaction Type', fontweight='bold')

df[df['isFraud']==0]['amount'].clip(upper=1e6).plot(
    kind='hist', bins=50, ax=axes[1], alpha=0.6, color='#0F1C2E', label='Legit')
df[df['isFraud']==1]['amount'].clip(upper=1e6).plot(
    kind='hist', bins=50, ax=axes[1], alpha=0.6, color='#C9A96E', label='Fraud')
axes[1].set_title('Amount Distribution', fontweight='bold')
axes[1].legend()

df['balance_drain'] = df['oldbalanceOrg'] - df['newbalanceOrig']
df.groupby('isFraud')['balance_drain'].mean().plot(
    kind='bar', ax=axes[2], color=['#0F1C2E','#C9A96E'])
axes[2].set_title('Avg Balance Drain by Fraud Label', fontweight='bold')
axes[2].set_xticklabels(['Legit','Fraud'], rotation=0)

plt.tight_layout()
plt.show()

## 3. Mann-Whitney U Statistical Tests

In [ ]:
fraud_df = df[df['isFraud']==1]
legit_df = df[df['isFraud']==0]

cols = ['amount','oldbalanceOrg','newbalanceOrig','oldbalanceDest','newbalanceDest']
print(f"{'Feature':<25} {'p-value':>12} {'Significant':>12}")
print("-"*52)
for col in cols:
    u, p = mannwhitneyu(fraud_df[col], legit_df[col], alternative='two-sided')
    sig = 'YES ***' if p < 0.001 else 'YES *' if p < 0.05 else 'NO'
    print(f"{col:<25} {p:>12.2e} {sig:>12}")

## 4. Graph Network Analysis

In [ ]:
sample = df[df['type'].isin(['TRANSFER','CASH_OUT'])].sample(min(5000, len(df)), random_state=42)

G = nx.DiGraph()
for _, row in sample.iterrows():
    G.add_edge(row['nameOrig'], row['nameDest'],
               amount=row['amount'], fraud=row['isFraud'])

print(f"Graph: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges")

in_deg = dict(G.in_degree())
top_receivers = sorted(in_deg.items(), key=lambda x: x[1], reverse=True)[:10]
print("\nTop 10 receiver accounts (potential money mules):")
for acc, deg in top_receivers:
    print(f"  {acc}: {deg} incoming transactions")

pagerank = nx.pagerank(G, max_iter=100)
top_pr = sorted(pagerank.items(), key=lambda x: x[1], reverse=True)[:5]
print("\nTop 5 by PageRank:")
for node, pr in top_pr:
    print(f"  {node}: {pr:.6f}")

## 5. Feature Engineering

In [ ]:
df2 = df.copy()
df2['balance_drain_orig'] = df2['oldbalanceOrg'] - df2['newbalanceOrig']
df2['balance_increase_dest'] = df2['newbalanceDest'] - df2['oldbalanceDest']
df2['amount_to_balance_ratio'] = df2['amount'] / (df2['oldbalanceOrg'] + 1)
df2['zero_balance_after'] = (df2['newbalanceOrig'] == 0).astype(int)
df2['exact_drain'] = (df2['balance_drain_orig'] == df2['amount']).astype(int)
df2 = pd.get_dummies(df2, columns=['type'], drop_first=True)
df2.drop(['nameOrig','nameDest','isFlaggedFraud'], axis=1, inplace=True)

X = df2.drop('isFraud', axis=1)
y = df2['isFraud']
print(f"Features: {X.shape[1]} | Fraud cases: {y.sum():,}")

## 6. Train Models

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_tr = scaler.fit_transform(X_train)
X_te = scaler.transform(X_test)

neg_pos_ratio = (y_train==0).sum() / (y_train==1).sum()

classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=500, class_weight='balanced'),
    'Random Forest':       RandomForestClassifier(200, class_weight='balanced', random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(200, random_state=42),
    'XGBoost':             XGBClassifier(scale_pos_weight=neg_pos_ratio, n_estimators=200,
                                          random_state=42, eval_metric='logloss'),
    'LightGBM':            LGBMClassifier(class_weight='balanced', n_estimators=200, random_state=42),
}

results = {}
for name, clf in classifiers.items():
    clf.fit(X_tr, y_train)
    pred = clf.predict(X_te)
    prob = clf.predict_proba(X_te)[:,1]
    results[name] = {'F1': f1_score(y_test, pred), 'AUC': roc_auc_score(y_test, prob)}
    print(f"{name:<25}  F1: {results[name]['F1']:.4f}  AUC: {results[name]['AUC']:.4f}")

best = max(results, key=lambda k: results[k]['AUC'])
print(f"\nBest model: {best}")

## 7. Business Impact

In [ ]:
best_clf = classifiers[best]
pred = best_clf.predict(X_te)
prob = best_clf.predict_proba(X_te)[:,1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y_test, pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Legit','Fraud'], yticklabels=['Legit','Fraud'])
axes[0].set_title('Confusion Matrix', fontweight='bold')

fpr, tpr, _ = roc_curve(y_test, prob)
axes[1].plot(fpr, tpr, color='#C9A96E', lw=2, label=f"AUC = {results[best]['AUC']:.4f}")
axes[1].plot([0,1],[0,1],'--',color='gray')
axes[1].set_title('ROC Curve', fontweight='bold')
axes[1].legend()
plt.tight_layout()
plt.show()

tp = cm[1][1]
avg_fraud = df[df['isFraud']==1]['amount'].mean()
print(f"Estimated fraud prevented: {tp * avg_fraud:,.0f} (transaction value)")
print(f"False positives (customer friction): {cm[0][1]:,}")

# Final Business Interpretation

## Key Findings

- Fraud detection should prioritise recall and F1 Score rather than accuracy because fraud is rare.
- Graph-based customer and account connectivity can expose suspicious transaction structures.
- Transaction amount, account movement and network behaviour are stronger indicators than raw transaction type alone.
- NetworkX features make the project stand out beyond a standard tabular classifier.
- False negatives carry high financial and regulatory cost in fraud workflows.

## Recommended Next Steps

- Use graph-based features to flag mule-account and money-ring behaviour.
- Set operating thresholds based on investigation capacity and fraud loss appetite.
- Deploy a tiered alerting workflow for high-risk transactions.
- Monitor feature drift as fraud patterns change over time.
- Add analyst feedback loops to improve model retraining.

## Interview Talking Point

A strong way to discuss this project in an interview:

> "Built a fraud detection workflow combining graph analytics and machine learning to identify suspicious transaction behaviour, prioritising F1 and recall for imbalanced financial crime detection."

## Production Considerations

Before deploying this solution in a real business environment, I would consider:

- Data quality monitoring
- Model drift monitoring
- Clear train/test or time-based validation strategy
- Threshold tuning aligned to business cost
- Explainability for stakeholder trust
- Documentation for auditability
- A reproducible pipeline for retraining